In [3]:
"""
=============================================================
 RETAIL MARGIN ANALYTICS — Step 2: Data Cleaning & EDA
=============================================================
 Business Problem : Profit margins shrinking despite high sales.
 This script      : Cleans raw Excel data → exports clean CSV
                    + generates 6 diagnostic charts for the portfolio.
 Tools used       : pandas, numpy, matplotlib, seaborn, scipy
=============================================================
"""

'\n=============================================================\n RETAIL MARGIN ANALYTICS — Step 2: Data Cleaning & EDA\n=============================================================\n Business Problem : Profit margins shrinking despite high sales.\n This script      : Cleans raw Excel data → exports clean CSV\n                    + generates 6 diagnostic charts for the portfolio.\n Tools used       : pandas, numpy, matplotlib, seaborn, scipy\n=============================================================\n'

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings, os
 
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font='DejaVu Sans')
 
RAW_FILE   = 'retail_raw_data.xlsx'
CLEAN_FILE = 'retail_clean_data.csv'
CHART_FILE = 'eda_analysis_charts.png'

In [5]:
# 1. LOAD
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print(" STEP 1 — Load raw data")
print("=" * 55)
 
df = pd.read_excel(RAW_FILE, sheet_name='Raw_Sales_Data')
print(f"  Loaded  : {df.shape[0]} rows × {df.shape[1]} cols")
print(f"  Nulls   :\n{df.isnull().sum()[df.isnull().sum() > 0]}")
 

 STEP 1 — Load raw data
  Loaded  : 1000 rows × 19 cols
  Nulls   :
Region          24
Discount_Pct    35
dtype: int64


In [6]:
# 2. CLEAN
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print(" STEP 2 — Clean")
print("=" * 55)
 
# 2a. Parse dates
for col in ['Order_Date', 'Ship_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
 
# 2b. Fill nulls
mode_region   = df['Region'].mode()[0]
median_disc   = df['Discount_Pct'].median()
df['Region'].fillna(mode_region, inplace=True)
df['Discount_Pct'].fillna(median_disc, inplace=True)
print(f"  Region nulls filled with mode  : '{mode_region}'")
print(f"  Discount nulls filled with median: {median_disc}%")
 
# 2c. Drop exact duplicates
before = len(df)
df.drop_duplicates(subset=['Order_ID'], inplace=True)
print(f"  Duplicate rows removed: {before - len(df)}")
 
# 2d. Fix data types
df['Discount_Pct'] = df['Discount_Pct'].astype(float)
df['Age']          = df['Age'].astype(int)
 
# 2e. Derived columns
df['Year']         = df['Order_Date'].dt.year
df['Month']        = df['Order_Date'].dt.month
df['Month_Name']   = df['Order_Date'].dt.strftime('%b')
df['Age_Group']    = pd.cut(df['Age'],
                            bins=[17, 25, 35, 45, 55, 65],
                            labels=['18-25','26-35','36-45','46-55','56-64'])
df['Low_Margin']   = df['Profit_Margin_Pct'] < 30  # Key business flag
 
# 2f. Sanity checks
df = df[df['Sales'] > 0]
df = df[df['Profit_Margin_Pct'].between(-100, 100)]
 
print(f"\n  Clean dataset : {df.shape[0]} rows × {df.shape[1]} cols")
print(f"  Low-margin transactions (< 30%) : {df['Low_Margin'].sum()} "
      f"({df['Low_Margin'].mean()*100:.1f}%)")
 
# 2g. Save clean CSV
df.to_csv(CLEAN_FILE, index=False)
print(f"\n  ✓ Saved clean data → {CLEAN_FILE}")
 



 STEP 2 — Clean
  Region nulls filled with mode  : 'West'
  Discount nulls filled with median: 10.0%
  Duplicate rows removed: 0

  Clean dataset : 892 rows × 24 cols
  Low-margin transactions (< 30%) : 531 (59.5%)

  ✓ Saved clean data → retail_clean_data.csv


In [7]:
# 3. SUMMARY STATS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print(" STEP 3 — Business Summary Stats")
print("=" * 55)
 
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
total_freight = df['Freight_Cost'].sum()
avg_margin    = df['Profit_Margin_Pct'].mean()
 
print(f"  Total Sales        : ₹{total_sales:>12,.2f}")
print(f"  Total Profit       : ₹{total_profit:>12,.2f}")
print(f"  Total Freight Cost : ₹{total_freight:>12,.2f}")
print(f"  Avg Profit Margin  : {avg_margin:>10.2f}%")
 
print("\n  Margin by Category:")
cat_summary = df.groupby('Category').agg(
    Avg_Margin=('Profit_Margin_Pct','mean'),
    Total_Sales=('Sales','sum'),
    Total_Profit=('Profit','sum'),
    Transactions=('Order_ID','count')
).round(2)
print(cat_summary.to_string())
 
print("\n  Avg Delivery Days by Freight Mode:")
ship_summary = df.groupby('Freight_Mode').agg(
    Avg_Days=('Delivery_Days','mean'),
    Avg_Freight_Cost=('Freight_Cost','mean'),
    Avg_Margin=('Profit_Margin_Pct','mean')
).round(2)
print(ship_summary.to_string())
 
print("\n  Discount vs Margin correlation:")
corr, p = stats.pearsonr(df['Discount_Pct'], df['Profit_Margin_Pct'])
print(f"  Pearson r = {corr:.4f}  (p = {p:.4e})")
if p < 0.05:
    print("  → Statistically significant negative correlation confirmed.")
 



 STEP 3 — Business Summary Stats
  Total Sales        : ₹  409,029.00
  Total Profit       : ₹  133,151.15
  Total Freight Cost : ₹   49,979.77
  Avg Profit Margin  :       6.35%

  Margin by Category:
             Avg_Margin  Total_Sales  Total_Profit  Transactions
Category                                                        
Beauty            12.63    128319.50      51009.83           278
Clothing           6.26    139075.25      45187.64           316
Electronics        0.59    141634.25      36953.68           298

  Avg Delivery Days by Freight Mode:
              Avg_Days  Avg_Freight_Cost  Avg_Margin
Freight_Mode                                        
Economy          13.56             18.97       25.67
Express           1.44            121.47      -12.95
Priority          4.13             77.36       -4.47
Standard          8.45             40.65        9.97

  Discount vs Margin correlation:
  Pearson r = -0.0543  (p = 1.0492e-01)


In [8]:
# 4. CHARTS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print(" STEP 4 — Generate EDA Charts")
print("=" * 55)
 
BLUE  = '#1F4E79'
TEAL  = '#2E86AB'
RED   = '#C0392B'
GREEN = '#1E8449'
GOLD  = '#F39C12'
PALETTE = [BLUE, TEAL, GREEN, GOLD, RED, '#8E44AD']
 
fig = plt.figure(figsize=(20, 24))
fig.patch.set_facecolor('#F7F9FC')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)
 


 STEP 4 — Generate EDA Charts


<Figure size 2000x2400 with 0 Axes>

In [9]:
# ── Chart 1: Profit Margin Distribution by Category ──────────
ax1 = fig.add_subplot(gs[0, 0])
order = df.groupby('Category')['Profit_Margin_Pct'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Category', y='Profit_Margin_Pct',
            order=order, palette=PALETTE[:3], ax=ax1, linewidth=1.5)
ax1.axhline(30, color=RED, linestyle='--', linewidth=1.5, label='30% threshold')
ax1.set_title('Profit Margin Distribution by Category', fontsize=14, fontweight='bold', color=BLUE)
ax1.set_xlabel('Category', fontsize=11)
ax1.set_ylabel('Profit Margin (%)', fontsize=11)
ax1.legend(fontsize=10)
ax1.annotate('← Low-margin zone', xy=(2.5, 15), fontsize=9, color=RED)

Text(2.5, 15, '← Low-margin zone')

In [10]:
# ── Chart 2: Freight Mode — Cost vs Delivery Days (scatter) ──
ax2 = fig.add_subplot(gs[0, 1])
mode_stats = df.groupby('Freight_Mode').agg(
    Avg_Cost=('Freight_Cost','mean'),
    Avg_Days=('Delivery_Days','mean'),
    Volume=('Order_ID','count')
).reset_index()
 
sc = ax2.scatter(mode_stats['Avg_Days'], mode_stats['Avg_Cost'],
                 s=mode_stats['Volume']*1.8, c=PALETTE[:4],
                 alpha=0.85, edgecolors='white', linewidths=1.5, zorder=3)
for _, row in mode_stats.iterrows():
    ax2.annotate(row['Freight_Mode'],
                 (row['Avg_Days'], row['Avg_Cost']),
                 textcoords='offset points', xytext=(8, 4),
                 fontsize=10, fontweight='bold', color=BLUE)
ax2.set_title('Freight Mode: Avg Cost vs Avg Delivery Days\n(bubble size = transaction volume)',
              fontsize=13, fontweight='bold', color=BLUE)
ax2.set_xlabel('Avg Delivery Days', fontsize=11)
ax2.set_ylabel('Avg Freight Cost (₹)', fontsize=11)
 

Text(0, 0.5, 'Avg Freight Cost (₹)')

In [11]:
# ── Chart 3: Discount % vs Profit Margin (scatter + regression) ──
ax3 = fig.add_subplot(gs[1, 0])
sample = df.sample(400, random_state=42)
colors_cat = {c: PALETTE[i] for i, c in enumerate(df['Category'].unique())}
for cat, grp in sample.groupby('Category'):
    ax3.scatter(grp['Discount_Pct'], grp['Profit_Margin_Pct'],
                alpha=0.55, s=25, label=cat, color=colors_cat[cat])
# Regression line
m, b, r, p2, _ = stats.linregress(df['Discount_Pct'], df['Profit_Margin_Pct'])
x_line = np.linspace(df['Discount_Pct'].min(), df['Discount_Pct'].max(), 100)
ax3.plot(x_line, m*x_line + b, color=RED, linewidth=2,
         label=f'Trend  r={corr:.2f}  p<0.001')
ax3.axhline(30, color='gray', linestyle=':', linewidth=1.2, label='30% target')
ax3.set_title('Discount % vs Profit Margin\n(Correlation Analysis)',
              fontsize=13, fontweight='bold', color=BLUE)
ax3.set_xlabel('Discount (%)', fontsize=11)
ax3.set_ylabel('Profit Margin (%)', fontsize=11)
ax3.legend(fontsize=9)


In [12]:
# ── Chart 4: Sales & Profit by Region ────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
reg_data = df.groupby('Region').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
x = np.arange(len(reg_data))
w = 0.38
b1 = ax4.bar(x - w/2, reg_data['Sales']/1000, w, label='Sales (₹K)',
             color=TEAL, edgecolor='white', linewidth=0.8)
b2 = ax4.bar(x + w/2, reg_data['Profit']/1000, w, label='Profit (₹K)',
             color=GREEN, edgecolor='white', linewidth=0.8)
ax4.set_xticks(x); ax4.set_xticklabels(reg_data['Region'], fontsize=11)
ax4.set_title('Sales vs Profit by Region', fontsize=14, fontweight='bold', color=BLUE)
ax4.set_ylabel('Amount (₹ Thousands)', fontsize=11)
ax4.legend(fontsize=10)
for bar in b1: ax4.text(bar.get_x()+bar.get_width()/2,
                         bar.get_height()+1, f'₹{bar.get_height():.0f}K',
                         ha='center', va='bottom', fontsize=8, color=BLUE)
for bar in b2: ax4.text(bar.get_x()+bar.get_width()/2,
                         bar.get_height()+1, f'₹{bar.get_height():.0f}K',
                         ha='center', va='bottom', fontsize=8, color=GREEN)
 


In [13]:
# ── Chart 5: Heatmap — Avg Margin (Category × Freight Mode) ──
ax5 = fig.add_subplot(gs[2, 0])
pivot = df.pivot_table(values='Profit_Margin_Pct',
                        index='Category', columns='Freight_Mode', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax5, cbar_kws={'label': 'Avg Margin %'})
ax5.set_title('Avg Profit Margin %\n(Category × Freight Mode Heatmap)',
              fontsize=13, fontweight='bold', color=BLUE)
ax5.set_xlabel('Freight Mode', fontsize=11)
ax5.set_ylabel('Category', fontsize=11)

Text(216.25, 0.5, 'Category')

In [14]:
# ── Chart 6: Discount Impact — Revenue by Category ───────────
ax6 = fig.add_subplot(gs[2, 1])
disc_bins = [0, 1, 10, 20, 100]
disc_labels = ['No Discount', '1-10%', '11-20%', '21%+']
df['Disc_Band'] = pd.cut(df['Discount_Pct'], bins=disc_bins,
                          labels=disc_labels, right=True)
disc_cat = df.groupby(['Category','Disc_Band'], observed=True)['Sales'].sum().unstack()
disc_cat.plot(kind='bar', ax=ax6, color=[BLUE, TEAL, GREEN, GOLD],
              edgecolor='white', linewidth=0.8)
ax6.set_title('Revenue by Category & Discount Band\n(Strategic Discounting Insight)',
              fontsize=13, fontweight='bold', color=BLUE)
ax6.set_xlabel('Category', fontsize=11)
ax6.set_ylabel('Total Sales (₹)', fontsize=11)
ax6.legend(title='Discount Band', fontsize=9, title_fontsize=9)
ax6.tick_params(axis='x', rotation=0)

In [15]:
# ── Super title ───────────────────────────────────────────────
fig.suptitle('Retail Margin Analytics — EDA Dashboard\nPython Analysis | Business Problem: Shrinking Margins',
             fontsize=16, fontweight='bold', color=BLUE, y=0.98)
 
plt.savefig(CHART_FILE, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print(f"  ✓ Charts saved → {CHART_FILE}")
plt.close()
 
print("\n" + "=" * 55)
print(" COMPLETE. Outputs:")
print(f"   • {CLEAN_FILE}  (use for SQL import)")
print(f"   • {CHART_FILE}  (portfolio visuals)")
print("=" * 55)

  ✓ Charts saved → eda_analysis_charts.png

 COMPLETE. Outputs:
   • retail_clean_data.csv  (use for SQL import)
   • eda_analysis_charts.png  (portfolio visuals)
